In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd drive/MyDrive/graph-fake-news-detection

/content/drive/MyDrive/graph-fake-news-detection


In [ ]:
import torch

TORCH, DEVICE = str(torch.__version__).split('+')

!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{TORCH}+{DEVICE}.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.6 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 54.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 89.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 95.4 MB/s eta 0:00:00


In [ ]:
from argparse import Namespace
from copy import deepcopy
from pathlib import Path

import pandas as pd
from src.utils.cv_hpo_pipeline import find_best_params

MODELS = ["bigcn", "gcnfn", "gnn", "gnncl"]
DATASETS = ["politifact", "gossipcop"]


PARAM_GRID_BY_DATASET = {
    "politifact": {
        "nhid": [64, 128],
        "batch_size": [32],
        "lr": [1e-3, 5e-3],
        "wd": [1e-3, 1e-2],
    },
    "gossipcop": {
        "nhid": [64, 128, 256],
        "batch_size": [512],
        "lr": [1e-3, 5e-3],
        "wd": [1e-3, 1e-2],
    },
}

args = Namespace(
    # --- chung ---
    device="cuda",          # tự chuyển sang CPU nếu không có GPU
    dataset="politifact",   # vòng lặp sẽ ghi đè
    feature="bert",         # UPFD có: profile / spacy / bert / content
    batch_size=32,          # sẽ bị grid ghi đè
    lr=1e-3,                # sẽ bị grid ghi đè
    weight_decay=1e-4,      # sẽ bị grid ghi đè qua key "wd"
    nhid=128,               # sẽ bị grid ghi đè
    num_workers=2,          # chỉ có tác dụng nếu pipeline đọc args.num_workers

    # --- gnn ---
    model = "sage",

    # --- bigcn ---
    TDdroprate=0.2,
    BUdroprate=0.2,

    concat=True,

    # --- gnncl ---
    max_nodes=None,         # None -> 500 (politifact) / 200 (gossipcop)
)


Path("results").mkdir(parents=True, exist_ok=True)
summary_path = Path("results") / "best_params_summary.csv"

best_by_run = {}   # {(model, dataset): best_params}
summary_rows = []

for model_name in MODELS:
    for dataset_name in DATASETS:
        print("\n" + "#" * 70)
        print(f"# {model_name.upper()} on {dataset_name}")
        print("#" * 70)

        run_args = deepcopy(args)
        run_args.dataset = dataset_name

        param_grid = deepcopy(PARAM_GRID_BY_DATASET[dataset_name])

        out = find_best_params(model_name, run_args, param_grid)

        best_by_run[(model_name, dataset_name)] = out["best_params"]
        summary_rows.append({
            "model": model_name,
            "dataset": dataset_name,
            **out["best_params"],
            "mean_f1": out["best_result"]["mean_f1"],
            "std_f1": out["best_result"]["std_f1"],
            "mean_accuracy": out["best_result"]["mean_accuracy"],
        })

        # Ghi summary sau mỗi run để không mất kết quả nếu bị ngắt giữa chừng
        pd.DataFrame(summary_rows).to_csv(summary_path, index=False)

        del out                          # giải phóng dataset đã load

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_path, index=False)
print(summary_df)


######################################################################
# BIGCN on politifact
######################################################################

DATASET
Full dataset   : 314
Development    : 93
Test           : 221
Features       : 768
Classes        : 2
Device         : cuda
Development labels: class 0 = 49, class 1 = 44
Namespace(device='cuda', dataset='politifact', feature='bert', batch_size=32, lr=0.001, weight_decay=0.0001, nhid=128, num_workers=2, model='sage', TDdroprate=0.2, BUdroprate=0.2, concat=True, max_nodes=None, num_features=768, num_classes=2)

HYPERPARAMETER SEARCH
Model          : bigcn
Configurations : 8
CV folds       : 5
Cached configs : 8

----------------------------------------------------------------------
Configuration 1/8
{'nhid': 64, 'batch_size': 32, 'lr': 0.001, 'wd': 0.001}
SKIPPED (already run): mean_f1=0.8716 +/- 0.0528

*** NEW BEST CONFIGURATION ***

----------------------------------------------------------------------
Configurat

/usr/local/lib/python3.13/dist-packages/torch_geometric/data/dataset.py:116: UserWarning: The `pre_transform` argument differs from the one used in the pre-processed version of this dataset. If you want to make use of another pre-processing technique, pass `force_reload=True` explicitly to reload the dataset.
  self._process()
/usr/local/lib/python3.13/dist-packages/torch_geometric/transforms/to_dense.py:42: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj = torch.sparse_coo_tensor(data.edge_index, edge_attr, size)



DATASET
Full dataset   : 314
Development    : 93
Test           : 221
Features       : 768
Classes        : 2
Device         : cuda
Development labels: class 0 = 49, class 1 = 44
Namespace(device='cuda', dataset='politifact', feature='bert', batch_size=32, lr=0.001, weight_decay=0.0001, nhid=128, num_workers=2, model='sage', TDdroprate=0.2, BUdroprate=0.2, concat=True, max_nodes=None, num_features=768, num_classes=2)

HYPERPARAMETER SEARCH
Model          : gnncl
Configurations : 8
CV folds       : 5
Cached configs : 0

----------------------------------------------------------------------
Configuration 1/8
{'nhid': 64, 'batch_size': 32, 'lr': 0.001, 'wd': 0.001}

Fold 1/5: train=74, val=19


RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [1, 500] but got: [1, 16000].